# T6



In [1]:
import numpy as np
from scipy.stats import chi2, norm

np.random.seed(42)


In [2]:
theta_true = 3.0
n = 100
beta = 0.95
alpha = 1 - beta

u = np.random.uniform(size=n)
x = (1 - u) ** (-1 / (theta_true - 1))

S = np.log(x).sum()
theta_mle = 1 + n / S

theta_mle


3.1863944109696107

## Точный доверительный интервал для медианы

Для распределения Парето

$$
F(x) = 1 - x^{-(\theta-1)}, \quad x \ge 1.
$$

Медиана:

$$
m = 2^{1/(\theta-1)}.
$$

Если положить $Y_i = \ln X_i$, то

$$
Y_i \sim Exp(\theta-1),
$$

поэтому

$$
2(\theta-1)\sum_{i=1}^n \ln X_i \sim \chi^2_{2n}.
$$


In [3]:
q1 = chi2.ppf(alpha / 2, df=2 * n)
q2 = chi2.ppf(1 - alpha / 2, df=2 * n)

median_hat = 2 ** (1 / (theta_mle - 1))
ci_median_exact = (
    2 ** (2 * S / q2),
    2 ** (2 * S / q1)
)

median_hat, ci_median_exact


(1.3730403466171544, (1.3008659826649698, 1.4764507308658756))

## Асимптотический доверительный интервал для $\theta$

Информация Фишера на одно наблюдение:

$$
I(\theta) = \frac{1}{(\theta-1)^2}.
$$

Значит

$$
\sqrt{n}(\hat\theta - \theta) \Rightarrow N(0, (\theta-1)^2).
$$

После подстановки оценки:


In [4]:
z = norm.ppf((1 + beta) / 2)

se_asym = (theta_mle - 1) / np.sqrt(n)
ci_theta_asym = (
    theta_mle - z * se_asym,
    theta_mle + z * se_asym
)

ci_theta_asym


(2.7578689808196004, 3.614919841119621)

## Bootstrap-интервалы для $\theta$

1. Параметрический bootstrap  
2. Непараметрический bootstrap


In [10]:
B = 10000

boot_param = np.empty(B)
for i in range(B):
    ub = np.random.uniform(size=n)
    xb = ub ** (-1 / (theta_mle - 1))
    boot_param[i] = 1 + n / np.log(xb).sum()

ci_boot_param = tuple(np.quantile(boot_param, [alpha / 2, 1 - alpha / 2]))
ci_boot_param


(2.8247589467420497, 3.685239290208046)

In [6]:
boot_nonparam = np.empty(B)
for i in range(B):
    xb = np.random.choice(x, size=n, replace=True)
    boot_nonparam[i] = 1 + n / np.log(xb).sum()

ci_boot_nonparam = tuple(np.quantile(boot_nonparam, [alpha / 2, 1 - alpha / 2]))
ci_boot_nonparam


(2.828876359567487, 3.687529126752082)

In [7]:
print("theta_true =", theta_true)
print("theta_mle =", round(theta_mle, 6))
print("median_hat =", round(median_hat, 6))
print("exact CI for median =", tuple(round(v, 6) for v in ci_median_exact))
print("asymptotic CI for theta =", tuple(round(v, 6) for v in ci_theta_asym))
print("parametric bootstrap CI =", tuple(round(v, 6) for v in ci_boot_param))
print("nonparametric bootstrap CI =", tuple(round(v, 6) for v in ci_boot_nonparam))


theta_true = 3.0
theta_mle = 3.186394
median_hat = 1.37304
exact CI for median = (1.300866, 1.476451)
asymptotic CI for theta = (2.757869, 3.61492)
parametric bootstrap CI = (2.806928, 3.683445)
nonparametric bootstrap CI = (2.828876, 3.687529)
